### Text Loader

In [1]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "Data_center_RAG/data",
    glob="**/*.txt",      
    loader_cls=TextLoader
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")

Loaded 10 documents


### Text Splitting

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Maximum size of each chunk
    chunk_overlap=50,  # Overlap between chunks to maintain context
    length_function=len,
    separators=[" "]
)
chunks = text_splitter.split_documents(documents)

### Embedding Models

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

## Initialize a simple Embedding model(no API Key needed!)
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


d:\PRAVEEN\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7493.86it/s]


### Intilialize the ChromaDB Vector Store And Stores the chunks in Vector Representation

In [5]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

persist_dir = "./chroma_db"

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(# while vector store intialization we will use the from_documents class
    documents=chunks,
    embedding=embedding_model,
    persist_directory=persist_dir   
)
print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_dir}")
print(vectorstore)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7461.37it/s]


Vector store created with 19 vectors
Persisted to: ./chroma_db


In [7]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

response = llm.invoke("Hello!")
print(response.content)

Hello. How can I help you today?


In [8]:
llm.invoke("What are the Rag pipelines means")

AIMessage(content="RAG (Retrieval, Augmentation, Generation) pipelines refer to a series of processes used in natural language processing (NLP) and artificial intelligence (AI) to generate human-like text or answers. The RAG pipeline is a framework for building conversational AI models that can retrieve relevant information, augment it with additional context, and generate coherent and accurate responses.\n\nHere's a breakdown of each stage in the RAG pipeline:\n\n1. **Retrieval (R)**: This stage involves retrieving relevant information from a knowledge base, database, or external sources. The goal is to identify the most relevant and accurate information related to the input query or prompt. This can be done using various techniques such as keyword extraction, named entity recognition, or semantic search.\n2. **Augmentation (A)**: In this stage, the retrieved information is augmented with additional context, such as definitions, explanations, or examples. This helps to provide a more 

### Modern RAG Chain Implementation

In [9]:
# Use these imports for stability
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

from langchain_core.prompts import ChatPromptTemplate

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [10]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001AF095812E0>, search_kwargs={'k': 3})

In [11]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate([
    ("system", system_prompt),
    ("human", "{input}")  # ✅ Lowercase "human"
])

In [12]:
document_chain = create_stuff_documents_chain(llm, prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001AF0BD06EA0>, async_client=<groq.resources.chat.comple

In [14]:
rag_chain = create_retrieval_chain(retriever, document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001AF095812E0>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don

In [15]:
response = rag_chain.invoke({"input": "How do I fix CrashLoopBackOff?"})
response

{'input': 'How do I fix CrashLoopBackOff?',
 'context': [Document(id='9cade721-50b0-4b07-88e6-47e09681e083', metadata={'source': 'Data_center_RAG\\data\\Runbooks\\kubernetes_runbook.txt'}, page_content='Runbook: CrashLoopBackOff Troubleshooting\n\nPurpose:\nThis runbook describes how to troubleshoot CrashLoopBackOff errors.\n\nProcedure:\n\nStep 1:\nCheck pod status.\n\nkubectl get pods\n\nStep 2:\nView logs.\n\nkubectl logs <pod-name>\n\nStep 3:\nCheck ConfigMaps and Secrets.\n\nStep 4:\nVerify image version.\n\nStep 5:\nRestart deployment.\n\nkubectl rollout restart deployment <deployment-name>\n\nExpected Result:\nPods should return to the Running state.'),
  Document(id='d7bfec1c-846a-45a6-82b6-81f6a5e4828e', metadata={'source': 'Data_center_RAG\\data\\Incident_Reports\\incident_003.txt'}, page_content='Incident ID: INC-003\n\nTitle:\nCrashLoopBackOff in Authentication Service\n\nDate:\n2026-06-25\n\nSeverity:\nMedium\n\nAffected Service:\nauthentication-service\n\nProblem:\nPods c

In [16]:
response['answer']

'To fix CrashLoopBackOff, follow the troubleshooting steps in the runbook, which include checking pod status, viewing logs, checking ConfigMaps and Secrets, verifying image version, and restarting the deployment. The specific steps can be executed using kubectl commands, such as "kubectl get pods" and "kubectl rollout restart deployment <deployment-name>". Additionally, common root causes like incorrect database passwords stored in Kubernetes Secrets should be investigated and corrected.'

In [17]:
from rag.retrieval import get_vectorstore
print(get_vectorstore()._collection.count())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7241.50it/s]


3675


In [18]:
from rag.retrieval import build_retriever
print(build_retriever().invoke("CrashLoopBackOff"))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8503.36it/s]


[Document(id='80f0506d-3afc-46a5-8e3f-4ac6fe91b620', metadata={'page': 5, 'author': 'B.P.U.T', 'creationdate': '2023-10-10T06:39:08+00:00', 'creator': 'Microsoft® Word 2016', 'page_label': '6', 'source': 'D:\\PRAVEEN\\uploads\\908736a2da33409e81f07f4645dd790d.pdf', 'producer': 'www.ilovepdf.com', 'total_pages': 92, 'moddate': '2023-10-10T06:39:10+00:00'}, page_content='is called graceful \ndegradation. Systems designed for graceful degradation are called fault tolerant.'), Document(id='b22d421b-199b-4b0c-b38c-d1436d70bbf8', metadata={'creationdate': '2023-10-10T06:39:08+00:00', 'author': 'B.P.U.T', 'moddate': '2023-10-10T06:39:10+00:00', 'total_pages': 92, 'page': 5, 'page_label': '6', 'source': 'D:\\PRAVEEN\\uploads\\f0b73f34c65a4a39b45789a5910bf9e1.pdf', 'creator': 'Microsoft® Word 2016', 'producer': 'www.ilovepdf.com'}, page_content='is called graceful \ndegradation. Systems designed for graceful degradation are called fault tolerant.'), Document(id='fa50f3d8-f124-43d1-9998-0c96cdd3

In [19]:
response = retriever.invoke(
    "How do I troubleshoot CrashLoopBackOff?"
)

for doc in response:
    print(doc.page_content)
    print("----------------")

Runbook: CrashLoopBackOff Troubleshooting

Purpose:
This runbook describes how to troubleshoot CrashLoopBackOff errors.

Procedure:

Step 1:
Check pod status.

kubectl get pods

Step 2:
View logs.

kubectl logs <pod-name>

Step 3:
Check ConfigMaps and Secrets.

Step 4:
Verify image version.

Step 5:
Restart deployment.

kubectl rollout restart deployment <deployment-name>

Expected Result:
Pods should return to the Running state.
----------------
Incident ID: INC-003

Title:
CrashLoopBackOff in Authentication Service

Date:
2026-06-25

Severity:
Medium

Affected Service:
authentication-service

Problem:
Pods continuously restarted.

Root Cause:
Incorrect database password stored in Kubernetes Secret.

Impact:
Users were unable to log in.

Resolution:
1. Updated Kubernetes Secret.
2. Restarted Deployment.
3. Verified application health.

Status:
Resolved
----------------
Server Restart SOP

1.
Notify operations.

2.
Verify backup completed.

3.
Gracefully stop services.

4.
Restart oper

In [20]:
from rag.pipeline import ask_question

print(ask_question("How to troubleshoot CrashLoopBackOff?"))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5288.45it/s]



========== FILTER ==========
{}

========== SOURCES ==========
f0b73f34c65a4a39b45789a5910bf9e1.pdf (Page 10)
93fc27b4ce86412ead9bb5d0fc541529.pdf (Page 10)
b07abe5693d94ae38787e72ee3934e71.pdf (Page 10)
e81328638b264efc84b93d9701aa402c.pdf (Page 10)
e86cf0c858fe41908e9b9beaba2aa6ef.pdf (Page 10)
{'answer': 'The information is not available in the uploaded project documents.', 'sources': ['f0b73f34c65a4a39b45789a5910bf9e1.pdf (Page 10)', '93fc27b4ce86412ead9bb5d0fc541529.pdf (Page 10)', 'b07abe5693d94ae38787e72ee3934e71.pdf (Page 10)', 'e81328638b264efc84b93d9701aa402c.pdf (Page 10)', 'e86cf0c858fe41908e9b9beaba2aa6ef.pdf (Page 10)']}


In [18]:
from rag.pipeline import retrieve_context

docs = retrieve_context(
    "How to troubleshoot CrashLoopBackOff?"
)

for doc in docs:
    print(doc.page_content)
    print("----------------")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2423.28it/s]


In [19]:
from rag.retrieval import get_vectorstore

db = get_vectorstore()
print(db._collection.count())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2274.30it/s]


0


In [21]:
import chromadb

client = chromadb.PersistentClient(
    path="chroma_db"
)


collection = client.get_collection(
    name="data_center_rag"
)


data = collection.get(
    limit=10,
    include=[
        "metadatas",
        "documents"
    ]
)


print(data["metadatas"])

for doc in data["documents"]:
    print("\n----")
    print(doc[:300])

[{'source': 'D:\\PRAVEEN\\uploads\\83b60432359141758fcfc99e0c8f2643.txt', 'discipline': 'Electrical', 'project': 'AI EPC Data Center', 'document_type': 'Procedure', 'filename': 'Electrical_Commissioning_Procedure.txt'}, {'document_type': 'Procedure', 'filename': 'Electrical_Commissioning_Procedure.txt', 'project': 'AI EPC Data Center', 'source': 'D:\\PRAVEEN\\uploads\\83b60432359141758fcfc99e0c8f2643.txt', 'discipline': 'Electrical'}, {'document_type': 'Procedure', 'discipline': 'Electrical', 'filename': 'Electrical_Commissioning_Procedure.txt', 'source': 'D:\\PRAVEEN\\uploads\\83b60432359141758fcfc99e0c8f2643.txt', 'project': 'AI EPC Data Center'}, {'discipline': 'Electrical', 'project': 'AI EPC Data Center', 'source': 'D:\\PRAVEEN\\uploads\\83b60432359141758fcfc99e0c8f2643.txt', 'filename': 'Electrical_Commissioning_Procedure.txt', 'document_type': 'Procedure'}, {'discipline': 'Electrical', 'filename': 'Electrical_Commissioning_Procedure.txt', 'source': 'D:\\PRAVEEN\\uploads\\a096c05